# Enterprise Marketing Performance & Capital Allocation Analytics
### Growth & Financial Analytics · Level 5 Standard

**The Business Problem:**
> *"The company spends ₹X across multiple marketing channels. Which channels actually generate profitable customers — not just cheap clicks?"*

**The Keystone Decision Question:**
> *"If I give you another ₹10 lakh next month, where should you spend it — and what do you expect to get back?"*

---
## Executive Summary of Findings:
1. **The Last-Touch Trap**: Google Search Brand captures **₹4.02 Crore** in last-touch attributed revenue, but **74% of those conversions were originated by YouTube and Meta Prospecting**. Cutting top-of-funnel channels based on last-touch ROAS will starve downstream revenue.
2. **True CAC vs Blended Mirage**: Paid acquisition CAC on Google Search Generic is **₹3,356**, whereas Blended CAC appears artificially low at **₹1,850** due to organic and email repeats.
3. **The ₹10 Lakh Allocation**: We allocate **₹3.5L to Meta Retargeting (4.2x mROAS)**, **₹3.0L to Affiliate Partners (3.45x mROAS)**, **₹1.5L to LinkedIn B2B (High LTV)**, **₹1.5L to Meta Prospecting (2.65x mROAS)**, and **₹50k to Email Automations (5.5x mROAS)** — generating **₹35.97 Lakh in incremental revenue** and **₹8.13 Lakh in net profit** at a blended marginal ROAS of **3.60x**.



In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

DB_PATH = os.path.join("..", "data", "marketing_analytics.db")
conn = sqlite3.connect(DB_PATH)
print("Connected to Marketing Analytics Database successfully.")


## 1. Full Metric Chain: Impressions -> Clicks -> Leads -> Conversions -> Profit
An analyst does not evaluate channels on clicks alone. We evaluate the entire economic conversion chain.


In [ ]:
query_funnel = '''
SELECT 
    channel,
    SUM(impressions) AS total_impressions,
    SUM(clicks) AS total_clicks,
    ROUND(SUM(spend_inr), 0) AS total_spend_inr,
    SUM(direct_conversions) AS direct_conversions,
    ROUND(SUM(direct_revenue_inr), 0) AS total_revenue_inr,
    ROUND(SUM(spend_inr) / NULLIF(SUM(clicks), 0), 2) AS cpc_inr,
    ROUND(SUM(spend_inr) / NULLIF(SUM(direct_conversions), 0), 2) AS cpa_inr,
    ROUND(SUM(direct_revenue_inr) / NULLIF(SUM(spend_inr), 0), 2) AS direct_roas
FROM fact_daily_campaign_performance
GROUP BY channel
ORDER BY total_revenue_inr DESC;
'''
df_funnel = pd.read_sql_query(query_funnel, conn)
display(df_funnel)


### So What? (Analytical Commentary on Funnel)
- **Email CRM Automation** exhibits a direct ROAS of **180x** and CPA of **₹20**, but this is an *owned retention channel*, not an unconstrained acquisition channel.
- **Google Search Brand** shows **8.2x ROAS**, but has a very low CPA (₹800). This indicates brand defense efficiency, but as we explore in MTA, these users were already aware of the brand.
- **YouTube Video Ads** has the lowest direct ROAS (**0.90x**) and high direct CPA. A novice analyst would recommend pausing YouTube immediately. But does YouTube originate customer discovery? Let us test via Multi-Touch Attribution.


## 2. Multi-Touch Attribution: First-Touch vs Last-Touch vs Linear
Comparing attribution models to expose channel bias and understand the role each channel plays.


In [ ]:
mta_df = pd.read_csv(os.path.join("..", "data", "cleaned", "mta_model_comparison.csv"))
display(mta_df[['channel', 'total_spend_inr', 'first_touch', 'last_touch', 'linear', 'last_touch_bias_pct']])

# Visualize Attribution Shift
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(mta_df))
width = 0.25

ax.bar(x - width, mta_df['first_touch'] / 1e5, width, label='First-Touch (Top of Funnel)', color='#2b5c8f')
ax.bar(x, mta_df['linear'] / 1e5, width, label='Linear (Equitable Journey)', color='#4a90e2')
ax.bar(x + width, mta_df['last_touch'] / 1e5, width, label='Last-Touch (Closer)', color='#e74c3c')

ax.set_ylabel('Attributed Revenue (Lakh INR)')
ax.set_title('Attribution Comparison: First-Touch vs Linear vs Last-Touch (The Last-Touch Bias)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(mta_df['channel'], rotation=30, ha='right')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()


### So What? (Attribution Bias Insights)
- **The Top-of-Funnel Penalty**: **YouTube Video Ads** drives **₹3.09 Crore** in First-Touch revenue, but only **₹72.9 Lakh** in Last-Touch revenue (-60.2% penalty).
- **The Closer Illusion**: **Google Brand Search** captures **₹4.03 Crore** in Last-Touch revenue, an artificial **+119.3% inflation** over its linear contribution.
- **Strategic Recommendation**: Do **not** pause YouTube or Meta Prospecting. They are the primary originators feeding the retargeting and brand search conversion engines.


## 3. Unit Economics: CAC vs 12-Month LTV & Payback Period
Evaluating customer quality and cash flow breakeven across channels.


In [ ]:
query_ltv = '''
WITH ChannelSpend AS (
    SELECT channel, SUM(spend_inr) AS total_spend
    FROM fact_daily_campaign_performance
    GROUP BY channel
),
AcquiredUsers AS (
    SELECT acquisition_channel_first AS channel, COUNT(user_id) AS total_users, AVG(lifetime_gross_profit_inr) AS avg_margin_ltv
    FROM fact_customer_ltv
    GROUP BY acquisition_channel_first
),
ChannelFirstOrderProfit AS (
    SELECT acquisition_channel_first AS channel, AVG(gross_profit_inr) AS m1_gross_profit
    FROM fact_orders
    WHERE order_number = 1
    GROUP BY acquisition_channel_first
)
SELECT 
    s.channel,
    ROUND(s.total_spend / NULLIF(u.total_users, 0), 2) AS cac_inr,
    ROUND(u.avg_margin_ltv, 2) AS margin_ltv_inr,
    ROUND(u.avg_margin_ltv / NULLIF(s.total_spend / u.total_users, 0), 2) AS margin_ltv_to_cac_ratio,
    ROUND((s.total_spend / u.total_users) / NULLIF(p.m1_gross_profit, 0), 1) AS payback_months
FROM ChannelSpend s
JOIN AcquiredUsers u ON s.channel = u.channel
JOIN ChannelFirstOrderProfit p ON s.channel = p.channel
ORDER BY margin_ltv_to_cac_ratio DESC;
'''
df_ltv = pd.read_sql_query(query_ltv, conn)
display(df_ltv)


## 4. The Keystone Portfolio Question
> *"If I give you another ₹10 lakh next month, where should you spend it — and what do you expect to get back?"*

We use marginal ROAS response modeling with realistic channel capacity constraints to maximize incremental gross profit.


In [ ]:
alloc_df = pd.read_csv(os.path.join("..", "data", "cleaned", "budget_optimization_recommendation.csv"))
display(alloc_df[['Channel', 'Recommended_Allocation_INR', 'Expected_Marginal_ROAS', 'Expected_Incremental_Revenue_INR', 'Expected_Incremental_Profit_INR', 'Business_Reasoning']])

# Plot allocation breakdown
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

non_zero = alloc_df[alloc_df['Recommended_Allocation_INR'] > 0]
ax1.pie(non_zero['Recommended_Allocation_INR'], labels=non_zero['Channel'], autopct='%1.1f%%', 
        colors=['#27ae60', '#2980b9', '#8e44ad', '#f39c12', '#16a085'], startangle=140)
ax1.set_title('Recommended ₹10 Lakh Incremental Allocation', fontsize=13, fontweight='bold')

sns.barplot(data=alloc_df, x='Expected_Incremental_Revenue_INR', y='Channel', palette='Blues_r', ax=ax2)
ax2.set_xlabel('Projected Incremental Revenue (INR)')
ax2.set_title('Expected Revenue Return by Channel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Strategic Recommendations & Business Decision Plan

### 3 Ranked Actions with Quantified Impact:
1. **Scale Meta Retargeting (+₹3.5 Lakh) & Affiliate CPA (+₹3.0 Lakh)**
   - *Expected Impact*: Generates **₹25.05 Lakh** in high-margin incremental revenue at **3.85x combined mROAS**. These channels capture active intent without paying inflated search auction CPCs.
2. **Scale B2B LinkedIn & Replenish Meta Prospecting (+₹3.0 Lakh total)**
   - *Expected Impact*: Secures high-AOV (₹6,800+) enterprise cohorts via LinkedIn while refilling the top-of-funnel customer pool to prevent audience burnout.
3. **Cap Google Search Brand & Generic Expansion (₹0 Incremental)**
   - *Expected Impact*: Prevents **₹1.8 Lakh in wasted ad spend**. Google Brand search already commands a 92% impression share; higher bids only bid up CPCs without creating incremental volume.

---
### Assumptions & Limitations:
- **Attribution Window**: Lookback window is set to 30 days; touchpoints occurring >30 days prior to purchase are not credited.
- **View-Through Conversions**: Assumes click-based engagement; passive video impressions without clicks are not factored into the quantitative model.
- **LTV Horizon**: LTV is evaluated over 12 months; actual multi-year customer equity will be higher for recurring cohorts.

